In [1]:
from bs4 import BeautifulSoup
from docx import Document
from docx.shared import Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
import os
import requests
import sqlite3

In [2]:
def init_database(db_name='allRecipes.db'):
    """
    This function initiate the database for every recipe.
    :param db_name: 
    :return: 
    """
    with sqlite3.connect(db_name) as con:
        cur = con.cursor()
        cur.execute("""
            CREATE TABLE IF NOT EXISTS recipes(
                ID INTEGER PRIMARY KEY,
                title TEXT,
                ingredients TEXT,
                guides TEXT,
                url TEXT,
                image_url TEXT
            )
        """)

In [3]:
init_database()

In [4]:
def getInfo(userRequest):
    """
    This function takes the user request and converts it to a link for further actions.
    :param userRequest: 
    :return: 
    """
    words = userRequest.split()
    if len(words) == 1:
        _url = f'https://www.delish.com/search/?q={words[0]}&type=Recipes'
    else:
        query = '+'.join(words)
        _url = f'https://www.delish.com/search/?q={query}&type=Recipes'
    return _url

In [5]:
def parseInfo(_url):
    """
    This function takes a link as a parameter and parses it into a BeautifulSoup object.
    :param _url: 
    :return: 
    """
    page = requests.get(_url)
    soup = BeautifulSoup(page.text, 'html.parser')
    return soup

In [6]:
def getPages(soup):
    """
    This function takes a BeautifulSoup object and returns a list of all pages.
    :param soup: 
    :return: 
    """
    urls = []
    for link in soup.find_all('a'):
        href = link.get('href')
        if href and 'cooking/recipe-ideas/' in href:
            urls.append(href)
    return urls[2:] 

In [7]:
def fetchRecipes(urls, max_urls):
    """
    This function takes a list of urls and fetches all the recipes into a list.
    :param urls: 
    :param max_urls: 
    :return: 
    """
    all_recipes = []

    for index, url_part in enumerate(urls):
        if max_urls is not None and index >= max_urls:
            break
        url = 'https://www.delish.com/search' + url_part
        page = requests.get(url)
        soup = BeautifulSoup(page.text, 'html.parser')
        
        title = soup.find('meta', {'name': 'title'}).get('content')
        
        ingredients = []
        for li in soup.find_all('li', class_='css-s5yyu3 e12sb1171'):
            ingredients.append(' '.join(li.stripped_strings))
        
        guides = []
        for li in soup.select('li.css-21v28f ol > li'):
            step_number = li.find('span', class_='e1241r8m0').get_text(strip=True)
            step_text = ' '.join(li.stripped_strings).replace(f'Step {step_number}', '').strip()
            guides.append(f'Step {step_number}: {step_text}')

        img_tag = soup.find('img', {'class': 'css-0 e1g79fud0'})
        image_url = img_tag.get('src') if img_tag else None

        all_recipes.append((
            index + 1,
            title,
            '\n'.join(ingredients),
            '\n'.join(guides),
            url,
            image_url
        ))
    
    return all_recipes

In [8]:
def save_to_database(recipes_data, db_name='allRecipes.db'):
    try:
        con = sqlite3.connect(db_name)
        cur = con.cursor()

        cur.executemany("""
            INSERT INTO recipes (ID, title, ingredients, guides, url, image_url)
            VALUES (?, ?, ?, ?, ?, ?)
        """, recipes_data)

        con.commit()
        print(f"Successfully inserted {len(recipes_data)} recipes")
    except sqlite3.Error as e:
        print(f"Database error occurred: {e}")
    finally:
        if con:
            con.close()

In [9]:
def download_recipe_images(all_recipes, max_images=None):
    """
    This function downloads all the images from all the recipes for further inserting.
    :param all_recipes: 
    :param max_images: 
    :return: 
    """
    for recipe in all_recipes:
        recipe_id = recipe[0]
        title = recipe[1]
        image_url = recipe[5]
        
        if max_images is not None and recipe_id > max_images:
            break
            
        if not image_url:
            print(f"No image URL for recipe {recipe_id}")
            continue

        sanitized_title = "".join(c if c.isalnum() else "_" for c in title)
        
        try:
            response = requests.get(image_url)
            if response.status_code == 200:
                content_type = response.headers.get('Content-Type', '').split(';')[0]
                extension_map = {
                    'image/jpeg': 'jpg',
                    'image/png': 'png',
                    'image/gif': 'gif',
                    'image/webp': 'webp'
                }
                extension = extension_map.get(content_type, 'bin')
                
                filename = f"recipe_{recipe_id}_{sanitized_title}.{extension}"
                
                with open(filename, 'wb') as f:
                    f.write(response .content)
                print(f"Saved image {filename}")
            else:
                print(f"Failed to download image from {image_url} with status code {response.status_code}")
        except Exception as e:
            print(f"An error occurred while downloading image from {image_url}: {e}")

In [10]:
def display_recipe_list():
    """
    This function shows the list of all recipes that have been requested.
    :return: 
    """
    with sqlite3.connect('allRecipes.db') as conn:
        cursor = conn.cursor()
        cursor.execute("""
            SELECT ID, title, url 
            FROM recipes 
            ORDER BY ID
        """)
        print("\nAvailable Recipes:")
        print("{:<5} {:<40} {:<50}".format("ID", "Title", "URL"))
        for row in cursor.fetchall():
            print("{:<5} {:<40} {:<50}".format(row[0], row[1][:60], row[2][:80]))   

In [11]:
def generate_recipe_doc(recipe_id):
    """
    This function generates DOCX file with the recipe that user have chosen.
    :param recipe_id: 
    :return: 
    """
    with sqlite3.connect('allRecipes.db') as conn:
        cursor = conn.cursor()
        cursor.execute("""
            SELECT * FROM recipes WHERE ID = ?
        """, (recipe_id,))
        
        recipe = cursor.fetchone()
        if not recipe:
            print("Recipe not found!")
            return
            
        recipe_id, title, ingredients, guides, url, image_url = recipe

        doc = Document()
        doc.add_heading(title, 0)

        doc.add_heading('Ingredients', level=1)
        for item in ingredients.split('\n'):
            doc.add_paragraph(item, style='List Bullet')
            
        doc.add_heading('Instructions', level=1)
        for step in guides.split('\n'):
            p = doc.add_paragraph(step)
            p.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY

        if image_url:
            try:
                sanitized_title = "".join(c if c.isalnum() else "_" for c in title)
                filename = f"recipe_{recipe_id}_{sanitized_title}.jpg"
                
                if not os.path.exists(filename):
                    response = requests.get(image_url)
                    response.raise_for_status()
                    with open(filename, 'wb') as f:
                        f.write(response.content)
                        
                doc.add_picture(filename, width=Inches(6))
            except Exception as e:
                print(f"Could not add image: {str(e)}")
        
        doc_name = f"Recipe_{recipe_id}_{sanitized_title}.docx"
        doc.save(doc_name)
        print(f"Document saved as {doc_name}")

In [12]:
def recipe_search_flow():
    """
    This function takes the user request, the number of recipes, and enables scraping.
    :return: 
    """
    search_term = input("Enter ingredient or recipe name: ")
    max_results = int(input("How many recipes to show? "))
    
    search_url = getInfo(search_term)
    search_soup = parseInfo(search_url)
    recipe_urls = getPages(search_soup) 
    recipes = fetchRecipes(recipe_urls, max_results)
    save_to_database(recipes)
    
    display_recipe_list()
    
    
    # selected_id = int(input("Enter recipe ID to generate document: "))
    # generate_recipe_doc(selected_id)

In [13]:
def start():
    recipe_search_flow()
    selected_id = int(input("Enter recipe ID to generate document: "))
    generate_recipe_doc(selected_id)

In [14]:
start()

Successfully inserted 5 recipes

Available Recipes:
ID    Title                                    URL                                               
1     Best Massaman Curry Recipe - How to Make Beef Massaman Curry https://www.delish.com/search/cooking/recipe-ideas/massaman-curry-recipe/
2     Best Vegetable Curry Recipe - How To Make Vegetable Curry https://www.delish.com/search/cooking/recipe-ideas/vegetable-curry-recipe/
3     Best Chicken Curry Noodle Soup Recipe - How To Make Chicken  https://www.delish.com/search/cooking/recipe-ideas/chicken-curry-noodle-soup-rec
4     Best Butternut Squash Curry Recipe - How to Make Squash Curr https://www.delish.com/search/cooking/recipe-ideas/butternut-squash-curry-recipe
5     Best Lamb Curry Recipe - How To Make Lamb Curry https://www.delish.com/search/cooking/recipe-ideas/lamb-curry-recipe/


KeyboardInterrupt: Interrupted by user